# ONS dataset exploration

This notebook downloads the ONS datasets relevant to the housing intelligence project, checks the available files, and profiles each dataset for schema, nulls, and categorical uniqueness.


In [ ]:
import os
from pathlib import Path
from typing import Any, Optional, Tuple

import pandas as pd
import requests

ONS_BASE = "https://api.beta.ons.gov.uk/v1"
DATASET_IDS = [
    "house-prices-local-authority",
    "index-private-housing-rental-prices",
    "mid-year-pop-est",
    "ashe-tables-7-and-8",
    "labour-market",
    "wellbeing-local-authority",
]
OUT_DIR = Path("ons_profiles")
OUT_DIR.mkdir(exist_ok=True)


In [ ]:
def get_json(url: str, params: Optional[dict[str, Any]] = None) -> Any:
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def get_latest_version_url(dataset_id: str) -> str:
    dataset_meta = get_json(f"{ONS_BASE}/datasets/{dataset_id}")
    latest = dataset_meta.get("links", {}).get("latest_version")
    if not latest:
        raise ValueError(f"No latest_version found for dataset: {dataset_id}")
    return latest["href"]


def get_download_url(version_url: str) -> tuple[str, str]:
    version_data = get_json(version_url)
    downloads = version_data.get("downloads", {})

    preferred_order = ["csv", "xlsx", "xls", "json"]
    for key in preferred_order:
        if key in downloads:
            entry = downloads[key]
            href = None
            if isinstance(entry, dict):
                href = entry.get("href") or entry.get("url")
            elif isinstance(entry, str):
                href = entry
            if href:
                return href, key.upper()

    for value in downloads.values():
        if isinstance(value, dict):
            href = value.get("href") or value.get("url")
            if href:
                return href, "FILE"

    raise ValueError(f"No downloadable file found for version: {version_url}")


def load_dataset_from_url(download_url: str) -> pd.DataFrame:
    try:
        return pd.read_csv(download_url, low_memory=False)
    except Exception:
        pass

    try:
        return pd.read_excel(download_url)
    except Exception:
        pass

    try:
        data = requests.get(download_url, timeout=60).json()
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            if "items" in data and isinstance(data["items"], list):
                return pd.DataFrame(data["items"])
            return pd.json_normalize(data)
    except Exception:
        pass

    raise ValueError(f"Could not load dataset from URL: {download_url}")


def describe_dataframe(dataset_id: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        s = df[column]
        rows.append(
            {
                "dataset_id": dataset_id,
                "column": column,
                "dtype": str(s.dtype),
                "null_count": int(s.isna().sum()),
                "null_pct": round(float(s.isna().mean() * 100), 2),
                "unique_count": int(s.nunique(dropna=True)),
                "unique_sample": ", ".join(map(str, s.dropna().astype(str).unique()[:5])),
            }
        )
    return pd.DataFrame(rows)


def summarize_categorical(df: pd.DataFrame):
    categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    if not categorical_cols:
        print("No categorical columns detected.")
        return

    print("\nCategorical summary:")
    for col in categorical_cols:
        s = df[col]
        print(f"\n- {col}")
        print(f"  nulls: {s.isna().sum()} | unique values: {s.nunique(dropna=True)}")
        value_counts = s.dropna().value_counts()
        for val, count in value_counts.head(10).items():
            print(f"    {val}: {count}")


337


In [ ]:
for dataset_id in DATASET_IDS:
    print(f"\n{'=' * 80}")
    print(f"DATASET: {dataset_id}")
    print(f"{'=' * 80}")

    try:
        version_url = get_latest_version_url(dataset_id)
        download_url, file_type = get_download_url(version_url)

        dataset_meta = get_json(f"{ONS_BASE}/datasets/{dataset_id}")
        print("Title:", dataset_meta.get("title"))
        print("Release frequency:", dataset_meta.get("release_frequency"))
        print("Last updated:", dataset_meta.get("last_updated"))
        print("Download format:", file_type)
        print("Download URL:", download_url)

        df = load_dataset_from_url(download_url)
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print("\nDtypes:")
        print(df.dtypes.to_string())
        print("\nNull counts:")
        print(df.isna().sum().to_string())

        print("\nPreview:")
        print(df.head(3).to_string(index=False))

        summarize_categorical(df)

        profile = describe_dataframe(dataset_id, df)
        out_path = OUT_DIR / f"{dataset_id}_profile.csv"
        profile.to_csv(out_path, index=False)
        print(f"\nSaved profile to: {out_path}")

    except Exception as exc:
        print(f"ERROR processing {dataset_id}: {exc}")


{'items': [{'contacts': [{'email': 'qualityoflife@ons.gov.uk', 'name': 'Will Shufflebottom, Owain Birrell, Pavan Bains and Geeta Kerai', 'telephone': '+44 300 0671543'}], 'description': 'Seasonally and non seasonally-adjusted quarterly estimates of life satisfaction, feeling that the things done in life are worthwhile, happiness and anxiety in the UK.', 'keywords': ['well-being'], 'id': 'wellbeing-quarterly', 'last_updated': '2023-12-13T09:40:24.204Z', 'links': {'editions': {'href': 'https://api.beta.ons.gov.uk/v1/datasets/wellbeing-quarterly/editions'}, 'latest_version': {'href': 'https://api.beta.ons.gov.uk/v1/datasets/wellbeing-quarterly/editions/time-series/versions/9', 'id': '9'}, 'self': {'href': 'https://api.beta.ons.gov.uk/v1/datasets/wellbeing-quarterly'}, 'taxonomy': {'href': 'https://api.beta.ons.gov.uk/v1/peoplepopulationandcommunity/wellbeing'}}, 'methodologies': [{'href': 'https://www.ons.gov.uk/peoplepopulationandcommunity/wellbeing/methodologies/personalwellbeingquarter